# Spatiotemporal Mobility Energy Load Modeling
### Metric: RMSE

**Key findings from EDA:**
- `seasonal_regime_code=3` in test is fully OOD (train has only {0,1,2})
- Regime is orthogonal to continuous features (R2=0.037 from PC1) — cannot be inferred
- Regime carries ~112 RMSE of independent signal, but that signal is unvalidatable for test
- PC1 of continuous features explains 98.8% of their joint variance
- `network_availability_flag=0` deterministically implies target=0 (verified on train)

**Regime strategy (choose one via `REGIME_STRATEGY`):**
- `'drop'`  — safe; loses 112 RMSE on OOF but no OOD extrapolation risk on test
- `'raw'`   — trees clip regime=3 to regime=2 leaf; NN extrapolates to ~1341; risky but higher OOF

In [1]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
pip('lightgbm'); pip('catboost'); pip('scikit-learn')

In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from scipy.optimize import minimize

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [3]:
PLATFORM_DATA_DIR = Path('dataset/public')
if PLATFORM_DATA_DIR.exists():
    TRAIN_PATH  = PLATFORM_DATA_DIR / 'train.csv'
    TEST_PATH   = PLATFORM_DATA_DIR / 'test.csv'
    SUBMIT_PATH = Path('working/submission.csv')
    Path('working').mkdir(exist_ok=True)
else:
    from google.colab import files
    files.upload()
    TRAIN_PATH  = Path('train.csv')
    TEST_PATH   = Path('test.csv')
    SUBMIT_PATH = Path('submission.csv')

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
print(f'Train: {train_raw.shape} | Test: {test_raw.shape}')

Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv
Train: (6576, 10) | Test: (2184, 10)


In [4]:
TARGET       = 'charging_demand_index'
NETWORK_FLAG = 'network_availability_flag'
SEED         = 42
N_FOLDS      = 10
N_PCS        = 6

# 'drop' or 'raw'
REGIME_STRATEGY = 'drop'

CONT_COLS = [
    'daily_activity_cycle_sin',
    'daily_activity_cycle_cos',
    'atmospheric_resistance_factor',
    'ambient_exposure_level',
    'precipitation_impact_index',
    'winter_disruption_index',
]

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE} | Regime strategy: {REGIME_STRATEGY}')

Device: cuda | Regime strategy: drop


In [5]:
# ── Feature engineering ────────────────────────────────────────────────────────
pca_model = PCA(n_components=N_PCS)
scaler    = StandardScaler()
scaler.fit(train_raw[CONT_COLS])
pca_model.fit(scaler.transform(train_raw[CONT_COLS]))

def build_features(df):
    X = df.copy()

    scaled   = scaler.transform(X[CONT_COLS])
    pcs      = pca_model.transform(scaled)
    for i in range(N_PCS):
      X[f'pc{i+1}'] = pcs[:, i]

    X['pc1_sq']  = X['pc1'] ** 2
    X['pc1_cb']  = X['pc1'] ** 3
    X['pc1_abs'] = np.abs(X['pc1'])

    X['event_x_pc1']  = X['special_event_flag'] * X['pc1']
    X['net_x_pc1']    = X[NETWORK_FLAG]         * X['pc1']

    X['env_sin_cos']  = X['daily_activity_cycle_sin'] * X['daily_activity_cycle_cos']
    X['atm_x_winter'] = X['atmospheric_resistance_factor'] * X['winter_disruption_index']
    X['precip_x_amb'] = X['precipitation_impact_index']    * X['ambient_exposure_level']
    X['cont_mag']     = np.sqrt((X[CONT_COLS] ** 2).sum(axis=1))

    if REGIME_STRATEGY == 'raw':
        # regime=3 in test is OOD; trees will use regime=2 leaf, NN will extrapolate
        X['regime_x_pc1']   = X['seasonal_regime_code'] * X['pc1']
        X['regime_x_pc1sq'] = X['seasonal_regime_code'] * X['pc1_sq']
        X['regime_sq']      = X['seasonal_regime_code'] ** 2
        X['event_x_regime'] = X['special_event_flag']   * X['seasonal_regime_code']
    else:
        X = X.drop(columns=['seasonal_regime_code'])

    return X


train_fe = build_features(train_raw)
test_fe  = build_features(test_raw)

FEATURE_COLS = [c for c in train_fe.columns if c not in [TARGET, 'id']]
print(f'Features: {len(FEATURE_COLS)}')

Features: 23


In [6]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def apply_network_mask(preds, flags):
    out = preds.copy()
    out[np.array(flags) == 0] = 0.0
    return out

X_all    = train_fe[FEATURE_COLS].values
y_all    = train_fe[TARGET].values
net_all  = train_raw[NETWORK_FLAG].values
X_test   = test_fe[FEATURE_COLS].values
net_test = test_raw[NETWORK_FLAG].values

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

In [7]:
# ── LightGBM ───────────────────────────────────────────────────────────────────
lgb_params = dict(
    objective='regression', metric='rmse',
    learning_rate=0.01, num_leaves=63, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8, subsample_freq=1,
    reg_alpha=0.1, reg_lambda=2.0,
    n_estimators=3000, n_jobs=-1, verbose=-1,
)

oof_lgb  = np.zeros(len(X_all))
test_lgb = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
    m = lgb.LGBMRegressor(**lgb_params, random_state=SEED+fold)
    m.fit(
        X_all[tr_idx], y_all[tr_idx],
        eval_set=[(X_all[va_idx], y_all[va_idx])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(False)]
    )
    oof_lgb[va_idx] = apply_network_mask(m.predict(X_all[va_idx]), net_all[va_idx])
    test_lgb       += apply_network_mask(m.predict(X_test), net_test) / N_FOLDS

lgb_rmse = rmse(y_all, oof_lgb)
print(f'LightGBM OOF RMSE: {lgb_rmse:.4f}')

LightGBM OOF RMSE: 418.7993


In [8]:
# ── CatBoost ───────────────────────────────────────────────────────────────────
cb_params = dict(
    loss_function='RMSE', learning_rate=0.01, iterations=3000,
    depth=7, l2_leaf_reg=5.0, min_data_in_leaf=20,
    verbose=False, task_type='GPU'
)

oof_cb  = np.zeros(len(X_all))
test_cb = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
    m = cb.CatBoostRegressor(**cb_params, random_state=SEED+fold)
    m.fit(X_all[tr_idx], y_all[tr_idx],
          eval_set=(X_all[va_idx], y_all[va_idx]),
          early_stopping_rounds=100)
    oof_cb[va_idx] = apply_network_mask(m.predict(X_all[va_idx]), net_all[va_idx])
    test_cb       += apply_network_mask(m.predict(X_test), net_test) / N_FOLDS

cb_rmse = rmse(y_all, oof_cb)
print(f'CatBoost  OOF RMSE: {cb_rmse:.4f}')

CatBoost  OOF RMSE: 418.7349


In [9]:
# ── Neural Network ─────────────────────────────────────────────────────────────
# ResNet MLP: input projection -> 4 residual blocks -> Softplus output
# Skip connections stabilize gradients across regimes; Softplus ensures positive predictions.
# With REGIME_STRATEGY='raw', the NN receives regime=3 and extrapolates continuously.

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(x + self.net(x))


class EnergyNet(nn.Module):
    def __init__(self, n_features, hidden=256, n_blocks=4):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU()
        )
        self.blocks = nn.Sequential(*[ResBlock(hidden) for _ in range(n_blocks)])
        self.output  = nn.Sequential(
            nn.Linear(hidden, 64), nn.SiLU(),
            nn.Linear(64, 1), nn.Softplus()
        )

    def forward(self, x):
        return self.output(self.blocks(self.input_proj(x))).squeeze(-1)

In [10]:
def make_loader(X, y=None, batch_size=1024, shuffle=False):
    Xt = torch.tensor(X, dtype=torch.float32)
    if y is not None:
        return DataLoader(TensorDataset(Xt, torch.tensor(y, dtype=torch.float32)),
                          batch_size=batch_size, shuffle=shuffle)
    return DataLoader(TensorDataset(Xt), batch_size=batch_size)


def train_nn(X_tr, y_tr, X_va, y_va, epochs=300, lr=5e-4, hidden=512, n_blocks=4):
    fs    = StandardScaler().fit(X_tr)
    model = EnergyNet(X_tr.shape[1], hidden=hidden, n_blocks=n_blocks).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.MSELoss()

    best_rmse  = 1e9
    best_state = {k: v.clone() for k, v in model.state_dict().items()}
    no_improve = 0
    tr_loader  = make_loader(fs.transform(X_tr), y_tr, shuffle=True)

    for epoch in range(epochs):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            crit(model(Xb), yb).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            Xv       = torch.tensor(fs.transform(X_va), dtype=torch.float32).to(DEVICE)
            val_rmse = rmse(y_va, model(Xv).cpu().numpy())
        if val_rmse < best_rmse:
            best_rmse  = val_rmse
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= 30:
            break

    model.load_state_dict(best_state)
    return model, fs


def predict_nn(model, fs, X):
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(fs.transform(X), dtype=torch.float32).to(DEVICE)
        return model(Xt).cpu().numpy()

In [11]:
oof_nn  = np.zeros(len(X_all))
test_nn = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
    model, fs = train_nn(
        X_all[tr_idx], y_all[tr_idx],
        X_all[va_idx], y_all[va_idx],
        epochs=300, lr=1e-3
    )
    oof_nn[va_idx] = apply_network_mask(predict_nn(model, fs, X_all[va_idx]), net_all[va_idx])
    test_nn       += apply_network_mask(predict_nn(model, fs, X_test), net_test) / N_FOLDS
    print(f'  Fold {fold+1}: {rmse(y_all[va_idx], oof_nn[va_idx]):.4f}')

nn_rmse = rmse(y_all, oof_nn)
print(f'\nNN OOF RMSE: {nn_rmse:.4f}')

  Fold 1: 400.3995
  Fold 2: 404.9137
  Fold 3: 404.7337
  Fold 4: 398.2996
  Fold 5: 429.4292
  Fold 6: 411.6366
  Fold 7: 391.1760
  Fold 8: 399.9985
  Fold 9: 430.7702
  Fold 10: 424.5862

NN OOF RMSE: 409.8078


In [12]:
print('=' * 42)
print(f'LightGBM  OOF RMSE : {lgb_rmse:.4f}')
print(f'CatBoost  OOF RMSE : {cb_rmse:.4f}')
print(f'NeuralNet OOF RMSE : {nn_rmse:.4f}')
print('=' * 42)

LightGBM  OOF RMSE : 418.7993
CatBoost  OOF RMSE : 418.7349
NeuralNet OOF RMSE : 409.8078


In [13]:
# ── Ensemble: OOF-optimized weights ───────────────────────────────────────────
oof_stack = np.stack([oof_lgb, oof_cb, oof_nn], axis=1)

def ensemble_rmse(w):
    w = np.abs(w) / np.abs(w).sum()
    return rmse(y_all, apply_network_mask((oof_stack * w).sum(axis=1), net_all))

res   = minimize(ensemble_rmse, x0=[1/3]*3, method='Nelder-Mead', options={'maxiter': 2000})
opt_w = np.abs(res.x) / np.abs(res.x).sum()
print(f'Weights — LGB: {opt_w[0]:.3f} | CB: {opt_w[1]:.3f} | NN: {opt_w[2]:.3f}')
print(f'Ensemble OOF RMSE: {ensemble_rmse(opt_w):.4f}')

Weights — LGB: 0.240 | CB: 0.065 | NN: 0.695
Ensemble OOF RMSE: 408.0729


In [14]:
# ── Full-data re-train (trees) ─────────────────────────────────────────────────
lgb_full = lgb.LGBMRegressor(**{**lgb_params, 'n_estimators': 5000})
lgb_full.fit(X_all, y_all, callbacks=[lgb.log_evaluation(False)])
pred_lgb_full = apply_network_mask(lgb_full.predict(X_test), net_test)

cb_full = cb.CatBoostRegressor(**{**cb_params, 'iterations': 5000})
cb_full.fit(X_all, y_all)
pred_cb_full = apply_network_mask(cb_full.predict(X_test), net_test)

test_stack  = np.stack([pred_lgb_full, pred_cb_full, test_nn], axis=1)
final_preds = np.clip(
    apply_network_mask((test_stack * opt_w).sum(axis=1), net_test),
    0, None
)
print(f'Predictions: min={final_preds.min():.2f}, mean={final_preds.mean():.2f}, max={final_preds.max():.2f}')

Predictions: min=0.00, mean=539.65, max=2360.53


In [15]:
submission = pd.DataFrame({'id': test_raw['id'], 'charging_demand_index': final_preds})
submission.to_csv(SUBMIT_PATH, index=False)
print(f'Saved: {SUBMIT_PATH}')
print(submission.head())

Saved: submission.csv
   id  charging_demand_index
0   0            1252.438656
1   1             168.138516
2   2             422.243487
3   3             897.348481
4   4             454.891694
